# Setup

In [ ]:
from pprint import pprint
from dotenv import dotenv_values


# Config
cfg = dotenv_values()
pprint(cfg)

OrderedDict({'CHAT_SMALL': 'qwen3:0.6b', 'SEED': '42'})


# 1. Prompt Template

## 1.1. PromptTemplate

### create

In [2]:
from pathlib import Path
from langchain_core.prompts import PromptTemplate
from langchain_core.load import dumps, loads


# Create
template = PromptTemplate.from_template(
    "My name is {name}\n"
    "I am from {nation}",

    name='user_prompt_template',
)

# Save & Load
path = Path('../tmp/prompt_template.json')
path.write_text(dumps(template))

template = loads(
    path.read_text(),
    allowed_objects=[PromptTemplate],   # explicitly specify.
)

# asdict
template.asdict()

# Print
template.pretty_print()
template.pretty_repr()

My name is {name}
I am from {nation}


C:\Users\aruwa\AppData\Local\Temp\ipykernel_19548\1888206857.py:18: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  template = loads(


'My name is {name}\nI am from {nation}'

### invoke

In [3]:
# Generate
inputs = {'name': 'yana', 'nation': 'korea'}
prompt = template.invoke(inputs)

# Generate, async
prompt = await template.ainvoke(inputs)

### partial

In [4]:
# Generate, partial
template = PromptTemplate.from_template(
    "You are a {role}.\n"
    "Answer this question: {question}"
)

# Set role once
prompt = template.partial(role="Python expert")

# Reuse with different questions
prompt.invoke({"question": "What is a tuple?"})
prompt.invoke({"question": "What is asyncio?"})

StringPromptValue(text='You are a Python expert.\nAnswer this question: What is asyncio?')

## 1.2. ChatPromptTemplate

In [5]:
from langchain_core.prompts import ChatPromptTemplate


messages = [
    ('system', 'You are a helpful assistant.'),
    ('human', 'My name is {name}'),
    ('placeholder', '{age}'),   # optional part
]
template = ChatPromptTemplate.from_messages(messages)

prompt = template.invoke({
    'name': 'yana',
    'age': ['18'],
})

prompt.messages

[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='My name is yana', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='18', additional_kwargs={}, response_metadata={})]

## 1.3. FewShotPromptTemplate

In [6]:
from langchain_core.prompts import PromptTemplate, FewShotPromptWithTemplates


# Examples
examples = [
    {'word': 'happy', 'opposite': 'sad'},
    {'word': 'hot', 'opposite': 'cold'},
]

example_template = PromptTemplate.from_template(
    "Word: {word}\n"
    "Opposite: {opposite}"
)

# Template
prefix = PromptTemplate.from_template(
    "You are a {role}. "
    "Give the opposite of each word."
)

suffix = PromptTemplate.from_template(
    "Word: {word}\nOpposite:"
)

template = FewShotPromptWithTemplates(
    examples=examples,
    example_prompt=example_template,
    prefix=prefix,
    suffix=suffix,
    input_variables=['role', 'word'],
)

# Prompt
inputs = {'role': 'dictionary', 'word': 'big'}
prompt = template.invoke(inputs)

print(prompt.text)

You are a dictionary. Give the opposite of each word.

Word: happy
Opposite: sad

Word: hot
Opposite: cold

Word: big
Opposite:


## 1.4. StructuredPrompt

In [7]:
from pydantic import BaseModel
from langchain_ollama import ChatOllama
from langchain_core.prompts.structured import StructuredPrompt


# Output structure
class Person(BaseModel):
    name: str
    nation: str

# Template: messages + output structure
template = StructuredPrompt(
    [
        ("system", "Extract the person's name and nation."),
        ("human", "{text}"),
    ],
    Person,
)


# Chat model
chat_model = ChatOllama(
    model=cfg['CHAT_SMALL']
)

# Chain: awares output structure
chat = template | chat_model

response = chat.invoke('My name is yana, and I am from Korea.')

response

C:\Users\aruwa\AppData\Local\Temp\ipykernel_19548\3643445783.py:12: LangChainBetaWarning: The class `StructuredPrompt` is in beta. It is actively being worked on, so the API may change.
  template = StructuredPrompt(


Person(name='Yana', nation='Korea')

# 2. Chat Model

## 2.1. Basic

- Model List: [here](https://ollama.com/search?c=thinking)

In [8]:
from langchain_ollama import ChatOllama


# Model
chat = ChatOllama(
    # Model
    model=cfg['CHAT_SMALL'],
    validate_model_on_init=True,    # ValidationError if model not exist
    reasoning=True,

    # Generation
    temperature=1.0,
    top_k=5,
    top_p=0.9,
    mirostat=2,             # Mirostat 2.0
    mirostat_tau=5.0,       # target mean perplexity
    mirostat_eta=0.1,       # mirostat learning rate
    stop=['</end>'],        # if generated, stop generation

    # Repeat
    repeat_last_n=64,       # check last n tokens
    repeat_penalty=1.1,     # penalize logits of repeated tokens

    # Input / Output
    num_ctx=2048,           # maxinum number of input tokens
    num_predict=512,        # maxinum number of generated tokens
    logprobs=True,          # if True, return logprobs of candidate tokens
    top_logprobs=3,         # number of candidate tokens to return logprobs

    # Runtime
    num_thread=None,        # ollama automatically detects the optimal number
    num_gpu=1,              # cuda:0
    keep_alive=None,        # how much does a model persists in memory after generation
    seed=cfg['SEED'],
)


# Response
response = chat.invoke("What is 1 + 1?")

print(f"--- Response ---\n{response.content}\n\n")
print(f"--- Reasoning ---\n{response.additional_kwargs['reasoning_content'][:50]}\n\n")

print("--- Log Probs ---")
for prob in response.response_metadata['logprobs'][:5]:
    print(f"{prob.token:<10s} {[(tok.token, round(tok.logprob, 2)) for tok in prob.top_logprobs]}")

--- Response ---
1 + 1 equals **2**.


--- Reasoning ---
Okay, the user is asking what 1 plus 1 is. Let me 


--- Log Probs ---
Okay       [('Okay', -0.05), ('Alright', -3.09), ('Well', -6.58)]
,          [(',', -0.0), (' so', -10.64), ('.', -11.0)]
 the       [(' the', -0.72), (' so', -0.75), (' let', -3.55)]
 user      [(' user', -0.04), (' question', -3.17), (' problem', -6.68)]
 is        [(' is', -0.03), (' asked', -3.74), (' asks', -5.11)]


## 2.2. Save & Load

In [9]:
from pathlib import Path

path = Path('../tmp/chat_config.json')

# Save
path.write_text(chat.model_dump_json())

# Load
chat = ChatOllama.model_validate_json(path.read_text())

## 2.3. Structured Output

In [10]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel


# Model
chat = ChatOllama(
    model=cfg['CHAT_SMALL'],
)

# Structured Output
class AnswerWithJustification(BaseModel):
    answer: str
    justification: str

chat_structured = chat.with_structured_output(AnswerWithJustification)

# Response
response = chat_structured.invoke("What is 2 * 3?")

print(f"[Answer]\n{response.answer}\n")
print(f"[Justification]\n{response.justification}")


[Answer]
6

[Justification]
2 * 3 = 6, as 2 + 2 + 2 = 6.


## 2.4. Streaming

In [11]:
from time import sleep


# Model
chat = ChatOllama(
    model=cfg['CHAT_SMALL'],
)

# Response
response = chat.stream("What is 2 * 3?")

for part in response:
    if part.content != '':
        print(part.content, end="", flush=True)
        sleep(0.1)

2 multiplied by 3 equals 6. 

**Answer:** 6

# 3. Chain

In [12]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import chain


# Chat model
chat = ChatOllama(
    model=cfg['CHAT_SMALL'],
)

# Template
messages = [
    ('system', "You are a helpful assistant."),
    ('human', "{query}"),
]
template = ChatPromptTemplate.from_messages(messages)


# Chain: template + chat
## 1. Chain: for complicated logics
@chain
def chatbot(values):
    prompt = template.invoke(values)
    response = chat.invoke(prompt)

    return response


## 2. LCEL: LangChain Expression Language
chatbot = template | chat


# Response
message = {'query': "What is 1 + 1?"}
response = chatbot.invoke(message)

pprint(response.content)

("In basic arithmetic, 1 + 1 equals **2**. If you're working with modular "
 'arithmetic (e.g., modulo 2), the result would be **0**. Let me know if '
 "you're exploring different mathematical concepts! 😊")
